# 04 — Lineage turnover and prediction

Do expanding lineages differ in composition from fading ones? Can composition predict subtype and year?

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)
from rsv import io, composition


In [ ]:
d = pd.read_csv("../data/processed/rsv_features.tsv", sep="\t")
FEATS = ["gc_content","freq_CG","freq_GC","freq_GG","freq_CC",
         "oe_CG","oe_GC","oe_GG","oe_CC"]

### Expanding vs fading lineages
Compare each lineage's share in an early window (2017–2020) vs a late window (2023–2026); label those gaining/losing >3 percentage points.

In [ ]:
def turnover(sub):
    x=d[d.Subtype==sub]
    early=x[x.year.between(2017,2020)].Lineage.value_counts(normalize=True)
    late =x[x.year.between(2023,2026)].Lineage.value_counts(normalize=True)
    chg=pd.DataFrame({"early":early,"late":late}).fillna(0)
    chg["delta"]=chg.late-chg.early
    comp=x.groupby("Lineage")[FEATS].mean()
    chg=chg.join(comp)
    chg["group"]=np.where(chg.delta>0.03,"expanding",np.where(chg.delta<-0.03,"fading","stable"))
    return chg
for s in "AB":
    t=turnover(s)
    g=t[t.group!="stable"].groupby("group")[["oe_CG","gc_content"]].mean().round(4)
    print(f"=== RSV-{s} ==="); print(g, "\n")

### Predict subtype and year from composition (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
sm = d.sample(min(8000,len(d)), random_state=1)
for tgt in ["Subtype","year"]:
    y=sm[tgt].astype(str)
    clf=RandomForestClassifier(n_estimators=200,n_jobs=-1,random_state=0)
    acc=cross_val_score(clf,sm[FEATS],y,cv=4).mean()
    base=y.value_counts(normalize=True).max()
    print(f"predict {tgt:8s}: acc={acc:.3f}  (majority baseline {base:.3f})")

In [ ]:
clf=RandomForestClassifier(n_estimators=300,n_jobs=-1,random_state=0)
clf.fit(sm[FEATS], sm["Subtype"])
imp=pd.Series(clf.feature_importances_,index=FEATS).sort_values()
imp.plot.barh(color="#6baed6"); plt.title("feature importance (subtype)"); plt.show()